# Grid-Guardian: Quickstart — Train, Evaluate & Pack

This notebook demonstrates the full Grid-Guardian RL training pipeline:
1. Load the partitioned dataset
2. Build environment, replay buffer & safety shield
3. Train BC baseline + CQL agent
4. Evaluate with safety metrics & stress tests
5. Run Offline Policy Evaluation (OPE)
6. Export model for Raspberry Pi deployment

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Dataset

In [ ]:
train_df = pd.read_csv("../data/partitioned/train.csv", nrows=5000)
val_df = pd.read_csv("../data/partitioned/val.csv", nrows=2000)

print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows")
print(f"Households: {train_df['household_id'].unique().tolist()}")
print(f"Columns: {list(train_df.columns[:10])}...")
train_df.head(3)

## 2. Build Environment & Replay Buffer

In [ ]:
from env.microgrid_env import MicrogridEnv, DISCRETE_ACTION_MAP
from env.safety_shield import SafetyShield, SafetyConfig
from data_utils.replay_buffer import DatasetConverter, ReplayBuffer

env_cfg = {
    "observation_keys": [
        "soc_kwh", "soc_capacity_kwh", "pv_gen_kw", "load_kw", "net_kw",
        "battery_power_kw", "price_signal",
        "forecast_irradiance_1h", "forecast_irradiance_3h", "forecast_temp_1h",
        "actual_irradiance_wm2", "voltage_v", "current_a",
    ],
    "time_features": True,
    "neighbor_balance": True,
    "action_type": "discrete",
    "safety": {"soc_min_frac": 0.10, "soc_max_frac": 0.95},
    "seed": 42,
}

env = MicrogridEnv(env_cfg, dataset=train_df, mode="replay")
eval_env = MicrogridEnv(env_cfg, dataset=val_df, mode="replay")

obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n
print(f"Obs dim: {obs_dim} | Act dim: {act_dim}")

converter = DatasetConverter(time_features=True, continuous=False)
trans = converter.convert(train_df)
buf = ReplayBuffer(capacity=len(trans['observations']) + 1000, seed=42)
buf.add_batch(trans)
print(f"Replay buffer: {len(buf)} transitions")
print(f"Episodes: {trans['dones'].sum()}")

## 3. Safety Shield

In [ ]:
shield = SafetyShield(SafetyConfig(shield_mode="clip"), DISCRETE_ACTION_MAP)
print(f"Safety shield: mode=clip")
print(f"SoC limits: [{shield.config.soc_min_frac*100:.0f}%, {shield.config.soc_max_frac*100:.0f}%]")
print(f"Max charge: {shield.config.max_charge_kw} kW | Max discharge: {shield.config.max_discharge_kw} kW")

## 4. Train BC Baseline

In [ ]:
from agents.bc_agent import BCAgent
from data_utils.replay_buffer import BehaviorDataset

bc = BCAgent(obs_dim, act_dim, [256, 256], 3e-4, False, "cpu")
bc_data = BehaviorDataset(time_features=True).build(train_df)
bc_hist = bc.train(bc_data, epochs=10, batch_size=256)
print(f"BC trained: final loss = {bc_hist['train_loss'][-1]:.6f}")

plt.figure(figsize=(8, 3))
plt.plot(bc_hist['train_loss'])
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('BC Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Train CQL Agent

In [ ]:
from agents.offline_rl import CQLAgent

cql = CQLAgent(obs_dim, act_dim, hidden=[256, 256], device="cpu")

losses = []
for step in range(500):
    batch = buf.sample(256)
    loss = cql.train_step(batch)
    losses.append(loss)
    if (step + 1) % 100 == 0:
        print(f"  Step {step+1}/500  loss={loss:.6f}")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('CQL Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluate Policy

In [ ]:
from evaluation.evaluator import evaluate_policy, stress_test

metrics = evaluate_policy(eval_env, cql.predict, n_episodes=5, shield=shield)

print(f"Mean reward:        {metrics['mean_reward']:.4f}")
print(f"Safety violation %: {metrics['safety_violation_rate']*100:.2f}%")
print(f"CVaR (5%):          {metrics['cvar_5pct']:.4f}")
print(f"SoC violations:     {metrics['soc_violations']}")
print(f"Energy sold (kWh):  {metrics['energy_sold_kwh']:.2f}")
print(f"Energy bought (kWh):{metrics['energy_bought_kwh']:.2f}")

## 7. Stress Tests

In [ ]:
stress = stress_test(eval_env, cql.predict,
                     scenarios=["cloud_ramp", "grid_outage", "sensor_dropout"],
                     n_episodes=2, shield=shield)

# Display as table
rows = []
for sc, m in stress.items():
    rows.append({
        "Scenario": sc,
        "Mean Reward": f"{m['mean_reward']:.2f}",
        "Safety Rate": f"{m['safety_violation_rate']*100:.1f}%",
        "SoC Violations": m['soc_violations'],
    })
pd.DataFrame(rows)

## 8. Offline Policy Evaluation (OPE)

In [ ]:
from evaluation.ope import run_ope

n_ope = min(3000, len(trans["observations"]))
ope_data = {k: v[:n_ope] for k, v in trans.items()}
print(f"OPE data: {n_ope} transitions, {ope_data['dones'].sum()} episodes")

ope_results = run_ope(cql.predict, bc, ope_data,
                      methods=["IS", "WIS"], gamma=0.99,
                      n_bootstrap=50, device="cpu",
                      eval_agent=cql)

for method, r in ope_results.items():
    print(f"  {method}: estimate={r['estimate']:.4f}  "
          f"CI=[{r['ci_lower']:.4f}, {r['ci_upper']:.4f}]  "
          f"n_episodes={r.get('n_episodes', 'N/A')}")

## 9. Export for Raspberry Pi

In [ ]:
from model_packaging.exporter import export_torchscript, export_onnx, NormalizationPipeline

norm = NormalizationPipeline.fit(trans["observations"])
os.makedirs("output", exist_ok=True)

try:
    export_torchscript(cql.q, obs_dim, "output/cql_policy.torchscript", norm)
    print("TorchScript exported: output/cql_policy.torchscript")
except Exception as e:
    print(f"TorchScript failed: {e}")

try:
    export_onnx(cql.q, obs_dim, "output/cql_policy.onnx", norm)
    print("ONNX exported: output/cql_policy.onnx")
except Exception as e:
    print(f"ONNX failed: {e}")

norm.save("output/norm_params.npz")
print("Normalization params: output/norm_params.npz")
print("\nModel ready for Raspberry Pi deployment!")
print("See edge/README.edge.md for deployment instructions.")

## 10. Edge Inference Test

In [ ]:
from edge.edge_inference import infer, safety_clip

# Simulate a single observation
sample_obs = trans["observations"][0]
result = infer("output/cql_policy.torchscript", sample_obs)
print(f"Action: {result['action_name']} ({result['action_kw']:.1f} kW)")
print(f"Logits: {[f'{x:.3f}' for x in result['logits']]}")

# Safety clip test
clipped = safety_clip(result['action_kw'], soc=0.08, soc_cap=10.0)
print(f"Safety clipped (low SoC): {clipped:.1f} kW")

---

**Next Steps:**
- Run full-scale training: `python train_rl_agents.py --config configs/train_rl.yaml --algo CQL --device cuda:0`
- Run all algorithms: `.\run_gpu_training.ps1`
- View TensorBoard: `tensorboard --logdir logs/`
- Deploy to Pi: see `edge/README.edge.md`